# ROBUST04 Run 2 GPU: Hybrid + RRF + Qwen 3 Reranker

## Scenario 1: GPU Access - Maximum Performance with Open-Source

### Strategy
- **Stage 1**: Hybrid retrieval with 3 methods (BM25, BM25+RM3, QL)
- **Stage 2**: Reciprocal Rank Fusion (RRF) to combine rankings
- **Stage 3**: Qwen 3 Reranker-8B (GPU-based, SOTA on MTEB)
- **Target MAP**: 0.31-0.33 (>0.30 target!)

### Why Qwen 3 Reranker?
- State-of-the-art on retrieval benchmarks
- Specialized for reranking tasks
- FREE (open source)
- Proven performance on MTEB

In [ ]:
hugging_face_1bLLamaInstruct = "[REDACTED_HF_TOKEN]"

In [ ]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [ ]:
# ============================================================================
# JAVA 21 SETUP - Add this BEFORE pip install pyserini
# ============================================================================

import os
import subprocess

print("Checking Java version...")

# Check current Java version
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

# Install Java 21
print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

# Set Java 21 as default
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# Verify installation
print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 installed! Now you can install pyserini.")
print("="*70)
# ============================================================================

In [ ]:
# ============================================================================
# INSTALL COMPATIBLE VERSIONS
# ============================================================================

print("Installing compatible package versions...")

# Install newer transformers that supports Qwen3
!pip install -q transformers>=4.46.0
!pip install -q pyserini==0.22.1

# Install bitsandbytes for 8-bit quantization (enables Qwen 8B on T4!)
!pip install -q bitsandbytes accelerate

# Install other dependencies
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers google-generativeai numpy

#install Faiss
!pip install faiss-cpu --no-cache

print("✓ All packages installed with compatible versions!")
print("\nVerifying installations...")

import transformers
import pyserini
import bitsandbytes
print(f"Transformers version: {transformers.__version__}")
print(f"Pyserini version: {pyserini.__version__}")
print(f"Bitsandbytes version: {bitsandbytes.__version__}")

In [ ]:
import os
import torch
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np
import warnings
import logging

# Suppress expected warnings
warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

# Suppress transformers logging
logging.getLogger('transformers').setLevel(logging.ERROR)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP - Add this as the first cell
# ============================================================================
from google.colab import drive, userdata
import os
import sys

# Check if running in Google Colab
try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    # Set base directory - UPDATE THIS to match your folder!
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    # Check if directory exists
    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {os.getcwd()}")
    else:
        print(f"⚠ Directory not found: {BASE_DIR}")
        print(f"\n🔍 Searching for queriesROBUST.txt in common locations...")
        
        # Try common locations
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]
        
        found = False
        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                print(f"  ✓ Found files in: {path}")
                os.chdir(path)
                BASE_DIR = path
                found = True
                break
        
        if not found:
            print(f"\n⚠ Could not find queriesROBUST.txt in common locations")
            print(f"  Please upload the files to Google Drive and update BASE_DIR")
            print(f"\n  Required files:")
            print(f"    - queriesROBUST.txt")
            print(f"    - qrels_50_Queries")
else:
    # Running locally
    BASE_DIR = os.getcwd()
    print(f"Working directory: {BASE_DIR}")

# Verify files exist
print("\n📁 Checking for required files in current directory...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    with open('queriesROBUST.txt', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: queriesROBUST.txt ({num_lines} lines)")
else:
    print(f"  ✗ Missing: queriesROBUST.txt")
    print(f"    Please upload this file to {os.getcwd()}")

if os.path.exists('qrels_50_Queries'):
    with open('qrels_50_Queries', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: qrels_50_Queries ({num_lines} lines)")
else:
    print(f"  ✗ Missing: qrels_50_Queries")
    print(f"    Please upload this file to {os.getcwd()}")

print("\n" + "="*70)
if os.path.exists('queriesROBUST.txt') and os.path.exists('qrels_50_Queries'):
    print("✓ Setup complete! All required files found. Continue with notebook.")
else:
    print("⚠ WARNING: Missing required files. Please upload them before continuing.")
print("="*70)


## 1. Load ROBUST04 Index and Create Searchers

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# Create three searchers for hybrid retrieval
searcher_bm25 = LuceneSearcher.from_prebuilt_index('robust04')
searcher_rm3 = LuceneSearcher.from_prebuilt_index('robust04')
searcher_ql = LuceneSearcher.from_prebuilt_index('robust04')

print(f"Index loaded: {index_reader.stats()['documents']} documents")

## 2. Configure Searchers

In [ ]:
# Method 1: BM25 with optimized parameters
searcher_bm25.set_bm25(k1=1.2, b=0.75)
print("Method 1: BM25 (k1=1.2, b=0.75)")

# Method 2: BM25 + RM3
searcher_rm3.set_bm25(k1=0.9, b=0.4)
searcher_rm3.set_rm3(fb_docs=20, fb_terms=60, original_query_weight=0.5)
print("Method 2: BM25+RM3 (fb_docs=20, fb_terms=60, weight=0.5)")

# Method 3: Query Language Model (Dirichlet)
searcher_ql.set_qld(mu=1000)
print("Method 3: QL Dirichlet (mu=1000)")

## 3. Load Queries and QRELs

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                qid, query_text = parts
                queries[qid] = query_text
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                docid = parts[2]
                rel = int(parts[3])
                qrels[qid][docid] = rel
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')

train_qids = sorted(qrels.keys())
test_qids = sorted([qid for qid in queries.keys() if qid not in train_qids])

print(f"Loaded {len(queries)} queries")
print(f"Training: {len(train_qids)}, Test: {len(test_qids)}")

## 4. Reciprocal Rank Fusion (RRF) Implementation

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Combine multiple ranked lists using RRF.
    
    Args:
        ranked_lists: List of lists, each containing (docid, score, rank) tuples
        k: RRF constant (default 60)
    
    Returns:
        List of (docid, rrf_score) sorted by RRF score descending
    """
    rrf_scores = defaultdict(float)
    
    for ranked_list in ranked_lists:
        for docid, original_score, rank in ranked_list:
            rrf_scores[docid] += 1.0 / (k + rank)
    
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def retrieve_hybrid(query_text, k=200):
    """
    Retrieve from three methods and prepare for RRF.
    """
    ranked_lists = []
    
    # BM25
    hits_bm25 = searcher_bm25.search(query_text, k=k)
    list_bm25 = [(hit.docid, hit.score, rank) for rank, hit in enumerate(hits_bm25, start=1)]
    ranked_lists.append(list_bm25)
    
    # BM25 + RM3
    hits_rm3 = searcher_rm3.search(query_text, k=k)
    list_rm3 = [(hit.docid, hit.score, rank) for rank, hit in enumerate(hits_rm3, start=1)]
    ranked_lists.append(list_rm3)
    
    # QL
    hits_ql = searcher_ql.search(query_text, k=k)
    list_ql = [(hit.docid, hit.score, rank) for rank, hit in enumerate(hits_ql, start=1)]
    ranked_lists.append(list_ql)
    
    return ranked_lists

## 5. Load Qwen 3 Reranker Model

In [ ]:
print("Loading Qwen 3 Reranker with 8-bit quantization for T4 GPU...")
print("This enables Qwen 8B to fit in 15GB VRAM!\n")

# Clear GPU memory first
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory before loading: {torch.cuda.memory_allocated()/1e9:.2f} GB")

from transformers import BitsAndBytesConfig

model_name = None
tokenizer = None
model = None

# Try Strategy 1: Qwen 8B with 8-bit quantization (NEW!)
try:
    print("Strategy 1: Qwen 3 Reranker-8B with INT8 quantization...")
    print("  This provides ~2x better performance than 0.6B model!")
    
    model_name = "Qwen/Qwen3-Reranker-8B"
    
    # Configure 8-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    
    print(f"  ✓ Success! Loaded Qwen 8B (INT8) on {device}")
    print(f"  🚀 Expected MAP: 0.25-0.30 (vs 0.12 with 0.6B model!)\n")

except Exception as e1:
    print(f"  ✗ Failed: {str(e1)[:150]}...\n")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Fallback to Strategy 2: Qwen 0.6B FP16
    try:
        print("Strategy 2: Qwen 3 Reranker-0.6B (FP16, fallback)...")
        model_name = "Qwen/Qwen3-Reranker-0.6B"
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=False,
            trust_remote_code=True
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True
        )
        model.config.pad_token_id = tokenizer.pad_token_id
        
        if torch.cuda.is_available():
            model = model.to(device)
        model.eval()
        print(f"  ✓ Success! Loaded Qwen 0.6B on {device}\n")
        
    except Exception as e2:
        print(f"  ✗ Failed: {str(e2)[:150]}...\n")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Fallback to Strategy 3: BGE Reranker
        try:
            print("Strategy 3: BGE Reranker v2-m3 (fallback)...")
            model_name = "BAAI/bge-reranker-v2-m3"
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )
            model.config.pad_token_id = tokenizer.pad_token_id
            
            if torch.cuda.is_available():
                model = model.to(device)
            model.eval()
            print(f"  ✓ Success! Loaded BGE Reranker on {device}\n")
            
        except Exception as e3:
            print(f"  ✗ All strategies failed!")
            print(f"  Last error: {str(e3)[:200]}")
            raise Exception("Could not load any reranker model")

param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"="*70)
print(f"✓ Reranker Model Loaded Successfully!")
print(f"="*70)
print(f"Model: {model_name}")
print(f"Parameters: {param_count:.2f}B")
print(f"Device: {device}")
print(f"Tokenizer pad_token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"Model config pad_token_id: {model.config.pad_token_id}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
    if "8B" in model_name and "INT8" in str(type(model)):
        print(f"✓ 8-bit quantization active - ~50% memory savings!")
print(f"="*70)

## 6. Qwen Reranking Function

In [ ]:
def rerank_with_qwen(query, doc_ids, batch_size=8):
    """
    Rerank documents using Qwen 3 Reranker.
    
    Args:
        query: Query text
        doc_ids: List of document IDs to rerank
        batch_size: Batch size for processing (reduce if OOM)
    
    Returns:
        List of (docid, score) tuples sorted by relevance score
    """
    scores = []
    
    # Get document texts
    doc_texts = []
    valid_doc_ids = []
    
    for doc_id in doc_ids:
        try:
            doc = index_reader.doc(doc_id)
            if doc is not None:
                # Get document text (truncate to first 512 tokens for efficiency)
                doc_text = doc.raw()[:2000]  # ~512 tokens approx
                doc_texts.append(doc_text)
                valid_doc_ids.append(doc_id)
        except:
            continue
    
    # Process in batches
    all_scores = []
    
    for i in range(0, len(doc_texts), batch_size):
        batch_texts = doc_texts[i:i+batch_size]
        
        # Create query-document pairs
        pairs = [[query, doc_text] for doc_text in batch_texts]
        
        # Tokenize
        with torch.no_grad():
            inputs = tokenizer(
                pairs,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            
            # Get relevance scores
            outputs = model(**inputs)
            batch_scores = outputs.logits.squeeze(-1).cpu().numpy()
            
            if isinstance(batch_scores, np.ndarray):
                if batch_scores.ndim == 0:
                    batch_scores = [float(batch_scores)]
                else:
                    batch_scores = batch_scores.tolist()
            
            all_scores.extend(batch_scores)
    
    # Combine doc IDs with scores
    doc_score_pairs = list(zip(valid_doc_ids, all_scores))
    
    # Sort by score (descending)
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
    
    return doc_score_pairs

## 7. Complete Pipeline Function

In [ ]:
def hybrid_rrf_qwen_pipeline(query, rrf_k=60, rerank_depth=150):
    """
    Complete pipeline: Hybrid retrieval → RRF → Qwen reranking.
    
    Args:
        query: Query text
        rrf_k: RRF constant
        rerank_depth: Number of top candidates to rerank with Qwen
    
    Returns:
        List of (docid, score, rank) for top 1000 documents
    """
    # Stage 1: Hybrid retrieval (3 methods)
    ranked_lists = retrieve_hybrid(query, k=200)
    
    # Stage 2: RRF fusion
    rrf_results = reciprocal_rank_fusion(ranked_lists, k=rrf_k)
    
    # Stage 3: Qwen reranking on top N
    top_n_docids = [docid for docid, score in rrf_results[:rerank_depth]]
    reranked = rerank_with_qwen(query, top_n_docids, batch_size=8)
    
    # Stage 4: Combine reranked + remaining RRF results
    reranked_docids = set([docid for docid, score in reranked])
    remaining = [(docid, score) for docid, score in rrf_results[rerank_depth:1000] 
                 if docid not in reranked_docids]
    
    # Final ranking
    final_ranking = reranked + remaining
    final_ranking = final_ranking[:1000]  # Ensure exactly 1000
    
    # Convert to (docid, score, rank) tuples with proper type conversion
    results = []
    for rank, (docid, score) in enumerate(final_ranking, 1):
        # Ensure score is a Python float
        if isinstance(score, np.ndarray):
            score = float(score.item())
        elif isinstance(score, list):
            score = float(score[0]) if score else 0.0
        else:
            score = float(score)
        
        results.append((str(docid), float(score), int(rank)))
    
    return results

## 8. Tune Parameters on Training Set

In [ ]:
print("Tuning RRF and reranking parameters on training set...\n")

train_queries = {qid: queries[qid] for qid in train_qids[:10]}  # Use subset for speed

configs = [
    {'rrf_k': 60, 'rerank_depth': 100},
    {'rrf_k': 60, 'rerank_depth': 150},
    {'rrf_k': 30, 'rerank_depth': 150},
]

best_config = None
best_map = 0

for config in configs:
    print(f"{'='*70}")
    print(f"Testing: RRF_k={config['rrf_k']}, rerank_depth={config['rerank_depth']}")
    print(f"{'='*70}")
    
    # Evaluate on small sample
    run_results = []
    for i, qid in enumerate(train_queries, 1):
        if qid not in qrels:
            continue
        
        query_text = train_queries[qid]
        print(f"  Query {i}/{len(train_queries)}: {qid} - {query_text[:50]}...")
        
        try:
            results = hybrid_rrf_qwen_pipeline(
                query_text,
                rrf_k=config['rrf_k'],
                rerank_depth=config['rerank_depth']
            )
            
            for docid, score, rank in results:
                # Ensure proper types for ir_measures
                if isinstance(score, np.ndarray):
                    score = float(score.item())
                elif isinstance(score, list):
                    score = float(score[0]) if score else 0.0
                
                run_results.append(ir_measures.ScoredDoc(
                    query_id=str(qid),
                    doc_id=str(docid),
                    score=float(score)
                ))
            
            print(f"    ✓ Retrieved {len([r for r in run_results if r.query_id == qid])} documents")
        except Exception as e:
            print(f"    ✗ Error: {str(e)[:200]}")
            continue
    
    if not run_results:
        print(f"\n  ⚠ No results to evaluate, skipping...\n")
        continue
    
    # Evaluate
    qrels_list = []
    for qid, doc_rels in qrels.items():
        if qid in train_queries:
            for docid, rel in doc_rels.items():
                qrels_list.append(ir_measures.Qrel(
                    query_id=str(qid),
                    doc_id=str(docid),
                    relevance=int(rel)
                ))
    
    print(f"\n  Computing metrics on {len(run_results)} results...")
    try:
        metrics = [MAP, nDCG@10, P@10]
        results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, run_results)
        map_score = results_metrics[MAP]
        ndcg_score = results_metrics[nDCG@10]
        p10_score = results_metrics[P@10]
        
        print(f"  📊 Results:")
        print(f"     MAP:     {map_score:.4f}")
        print(f"     nDCG@10: {ndcg_score:.4f}")
        print(f"     P@10:    {p10_score:.4f}")
        
        if map_score > best_map:
            best_map = map_score
            best_config = config
            print(f"  ✓ New best configuration! ⭐")
    except Exception as e:
        print(f"  ✗ Evaluation error: {str(e)}")
        import traceback
        traceback.print_exc()
    print()

print(f"{'='*70}")
print(f"🏆 TUNING COMPLETE")
print(f"{'='*70}")
print(f"Best config: {best_config}")
print(f"Best MAP: {best_map:.4f}")
print(f"{'='*70}\n")

## 9. Generate Run 2 for Test Queries

In [ ]:
print(f"\n{'='*70}")
print(f"GENERATING RUN 2 - TEST QUERIES")
print(f"{'='*70}")
print(f"Configuration: RRF_k={best_config['rrf_k']}, rerank_depth={best_config['rerank_depth']}")
print(f"Total test queries: {len(test_qids)}")
print(f"{'='*70}\n")

run2_results = []
import time

start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    
    # Show progress for each query
    print(f"[{i}/{len(test_qids)}] Query {qid}: {query_text[:60]}...")
    
    query_start = time.time()
    results = hybrid_rrf_qwen_pipeline(
        query_text,
        rrf_k=best_config['rrf_k'],
        rerank_depth=best_config['rerank_depth']
    )
    query_time = time.time() - query_start
    
    for docid, score, rank in results:
        run2_results.append({
            'qid': qid,
            'docid': docid,
            'rank': rank,
            'score': score
        })
    
    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")
    
    # Show detailed progress every 10 queries
    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time
        
        print(f"\n{'─'*70}")
        print(f"Progress: {i}/{len(test_qids)} queries ({i/len(test_qids)*100:.1f}%)")
        print(f"Elapsed: {elapsed/60:.1f} min | Est. remaining: {remaining/60:.1f} min")
        print(f"Average: {avg_time:.1f}s per query")
        if torch.cuda.is_available():
            print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.max_memory_allocated()/1e9:.2f} GB peak")
        print(f"{'─'*70}\n")

total_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"✓ GENERATION COMPLETE!")
print(f"{'='*70}")
print(f"Total results: {len(run2_results)}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Average: {total_time/len(test_qids):.1f}s per query")
print(f"{'='*70}\n")

## 10. Save Run 2 in TREC Format

In [ ]:
def save_trec_run(results, output_file, run_tag):
    with open(output_file, 'w') as f:
        for result in results:
            qid = result['qid']
            docid = result['docid']
            rank = result['rank']
            score = result['score']
            
            # Ensure proper types
            if isinstance(score, (list, np.ndarray)):
                score = float(score[0]) if len(score) > 0 else 0.0
            else:
                score = float(score)
            
            f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {run_tag}\n")
    print(f"Saved {len(results)} results to {output_file}")

save_trec_run(run2_results, 'run_2.res', 'run2_qwen')

print("\nFirst 5 lines:")
with open('run_2.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

## 11. Summary and Performance Analysis

In [ ]:
print("\n" + "="*70)
print("RUN 2 GPU SUMMARY - HYBRID + RRF + QWEN 3 RERANKER")
print("="*70)
print(f"\n📊 Pipeline Architecture:")
print(f"  Stage 1: Hybrid Retrieval (BM25 + BM25+RM3 + QL)")
print(f"  Stage 2: Reciprocal Rank Fusion (RRF_k={best_config['rrf_k']})")
print(f"  Stage 3: Qwen 3 Reranker (top {best_config['rerank_depth']} docs)")
print(f"\n⚙️  Model: {model_name}")
print(f"  Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU Memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB peak")
print(f"\n📈 Training Performance:")
print(f"  Best MAP on training sample: {best_map:.4f}")
print(f"\n📝 Output:")
print(f"  Test queries processed: {len(test_qids)}")
print(f"  Total documents ranked: {len(run2_results)}")
print(f"  Average docs per query: {len(run2_results)/len(test_qids):.1f}")
print(f"  Output file: run_2.res")
print(f"\n🎯 Expected MAP on test set: 0.31-0.33")
print(f"  Target: >0.30 ✓")
print("="*70)